In [1]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
import math
from functools import reduce

In [4]:
# ============================================================
# Code to compute gound state density matrix for a plaquette system
# ============================================================

# -------------------------
# Tensor product helper
# -------------------------
def kron_power(v, N):
    """Compute v ⊗ v ⊗ ... ⊗ v (N times)"""
    return reduce(np.kron, [v] * N)


# -------------------------
# Basis states |0> and |1>
# -------------------------
ket_0 = np.array([1, 0])
ket_1 = np.array([0, 1])



# -------------------------
# Pauli Matrices and Identity
# -------------------------

#Pauli Matrices

Pauli_X= np.array([[0, 1],[1, 0]])
Pauli_Z= np.array([[1, 0],[0, -1]])

# identity matrix 

I = np.eye(2)


#Defining local operator
def local_operator(op, i, N):
    """
    Build operator acting on i-th qubit (0-based index) in N-qubit system.
    """
    ops = [I] * N
    ops[i] = op
    return reduce(np.kron, ops)

#Defining plaquette operat


# ============================================================
# Evaluating gound state density matrix for 4 plaquettes system
# ============================================================

#Total qubit number 
N = 12


# -------------------------
# Build product states
# |0...0> and |1...1>
# -------------------------
ket_0N = kron_power(ket_0, N)
ket_1N = kron_power(ket_1, N)

def plaquette_operator(i1,i2,i3,i4,N):
    return local_operator(Pauli_X, i1, N) @ local_operator(Pauli_X, i2, N) @ local_operator(Pauli_X, i3, N) @ local_operator(Pauli_X, i4, N)

ground_state = 1/2**4 * ( (kron_power(I, N)+ plaquette_operator(0,1,2,3,N)) @ (kron_power(I, N)+ plaquette_operator(3,4,5,6,N)) @ (kron_power(I, N)+ plaquette_operator(6,7,8,9,N)) @ (kron_power(I, N) + plaquette_operator(9,10,11,1,N)) @ ket_0N)

print(ground_state)

[0.0625 0.     0.     ... 0.     0.     0.    ]


In [5]:
rho_0 = np.outer(ground_state, ground_state.conj())
print(rho_0)

[[0.00390625 0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


In [ ]:
# ============================================================
# Evaluating gound state density matrix for 4 plaquettes system
# ============================================================

#Total qubit number 
N = 12

In [ ]:
# ============================================================
# Code to compute density matrix after noise channel application
# ============================================================

In [ ]:
# ============================================================
# Code to compute entropies and CMI for given density matrix
# ============================================================

#Calculating von Neumann entropy
def von_neumann_entropy(rho):
    """
    Compute von Neumann entropy S(rho) = -Tr(rho log rho) = -Sum (lamba log lamba), where lambda's are eigenvalues
    """
    eigvals = np.linalg.eigvalsh(rho)

    # remove zeros safely
    eigvals = eigvals[eigvals > 1e-12]

    S = -np.sum(eigvals * np.log2(eigvals))

    return S


#Function to compute reduced density matrix
def partial_trace(rho, keep, N):
    """
    Partial trace over all qubits not listed in keep.

    rho:
        Full density matrix of shape (2^N, 2^N)

    keep:
        List of qubit indices to keep, using Python 0-based indexing.

    N:
        Total number of qubits.

    Returns:
        Reduced density matrix on the qubits in keep.
    """

    keep = sorted(keep)

    # Qubits to trace out
    trace_out = [q for q in range(N) if q not in keep]

    # Reshape rho into tensor form:
    # rho[i1, i2, ..., iN, j1, j2, ..., jN]
    rho_tensor = rho.reshape([2] * N + [2] * N)

    # Trace out qubits from largest index to smallest.
    # This avoids axis-shifting problems after each trace.
    current_N = N

    for q in sorted(trace_out, reverse=True):
        rho_tensor = np.trace(
            rho_tensor,
            axis1=q,
            axis2=q + current_N,
        )
        current_N -= 1

    dim_keep = 2 ** len(keep)

    return rho_tensor.reshape(dim_keep, dim_keep)


#Function to compute CMI depending on noise rate
def CMI(p, rho_0, A, B, C, N):
    """
    Compute I(A:C|B) = S(AB) + S(BC) - S(B) - S(ABC).
    """

    rho = noise_channel(rho_0, p, N)

    AB = sorted(A + B)
    BC = sorted(B + C)
    ABC = sorted(A + B + C)

    S_AB = von_neumann_entropy(partial_trace(rho, AB, N))
    S_BC = von_neumann_entropy(partial_trace(rho, BC, N))
    S_B = von_neumann_entropy(partial_trace(rho, B, N))
    S_ABC = von_neumann_entropy(partial_trace(rho, ABC, N))

    return S_AB + S_BC - S_B - S_ABC